# 12 RAG项目Skills与Agent迁移及Git提交


## 1. 对应课程与本篇范围

上一篇根据共享工程整理了 RAG 项目的单元测试与 BUG 调试。本篇继续把这些检查接入开发流程：将记账项目中已经建立的 Skills、专项 Agent 和提交质量门迁移到 RAG 项目，使测试、审查和 Git 提交使用同一套项目约定。

对应网盘视频：

```text
23-RAG项目-迁移技能和Agent及git提交.mp4
```

课程资料见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`；配套讲义为 [Part3 LangChain RAG项目实战](https://www.yuque.com/xxcls/vibecoding/mtwqfc61tifo9cgi)。本篇的主要依据是网盘 `项目/LangChainRAG项目/.claude/` 中实际保存的技能、代理、脚本与配置，以及根目录 `.gitignore`。

讲义仍以项目初始化提示词为主，没有本节完整迁移对话和 Git 操作记录。视频未逐段观看或转录，因此下文解释的是**共享工程中已经迁移的文件及其工作流程**，不编造课堂执行顺序、测试通过次数、提交哈希或远端地址。新增的操作建议明确标为“补充”。

本篇承接 07—09 的工程化机制和 11 的测试范围；下一节的并发压力测试另行整理。


## 2. 迁移哪些内容：流程可以复用，项目上下文要重写

前一个应用是记账桌面项目；当前应用是 FastAPI + React 的电商 RAG 知识库问答系统。迁移的价值在于保留已经整理好的工作步骤，同时让步骤指向当前工程的语言、目录和运行入口。

| 层次 | 原有职责 | RAG 项目中的适配重点 |
|---|---|---|
| Skill | 保存某类任务的步骤 | `unit-test` 同时覆盖 pytest 与 Vitest；安全检查加入 JWT、上传和 RAG 管道 |
| 专项 Agent | 组织角色的工作与输出 | `tester` 处理前后端测试；`quality-engineer` 使用两项审计技能并检查双端代码质量 |
| 提交 Agent | 依次组织测试、审查和提交 | `gitcommit-agent` 汇总两份结果，决定是否进入 Git 提交流程 |
| Hook | 在工具执行前检查条件 | 对命中的提交操作读取当前项目的结果文件 |
| 辅助脚本 | 用程序表达放行条件 | 两份结果齐全、首行符合约定时返回 0；失败返回 2 |
| 运行技能 | 保存项目启动与重建操作 | 从桌面应用入口调整为 Python 后端、Vite 前端和两套运行数据 |

共享代理已明确当前技术栈：后端使用 FastAPI、Python、LangChain、SQLAlchemy 和 ChromaDB；前端使用 React 18、TypeScript、Vite、Ant Design 5 和 Zustand；模型通过百炼 DashScope 调用，关系数据使用 SQLite。

迁移后还需要核对名称之间的引用：Agent 调用的技能名、提交代理指定的角色名、Hook 读取的结果文件名，必须与实际文件和 frontmatter 对应。只改角色介绍，原目录和命令仍然会把工作带到旧项目。


## 3. 共享工程中实际有哪些文件

网盘 RAG 项目的 `.claude/` 目录包含以下结构。`results/` 中两个文件名来自代理与 Hook 的约定；本次没有把该目录中的历史内容当作检查结果。

```text
LangChainRAG项目/
├── .gitignore
├── backend/
├── frontend/
└── .claude/
    ├── skills/
    │   ├── unit-test/SKILL.md
    │   ├── security-audit/SKILL.md
    │   ├── comments-check/SKILL.md
    │   ├── run-app/SKILL.md
    │   └── rebuild-app/SKILL.md
    ├── agents/
    │   ├── tester.md
    │   ├── quality-engineer.md
    │   └── gitcommit-agent.md
    ├── scripts/
    │   ├── check-quality-gate.sh
    │   └── check-quality-gate.ps1
    ├── results/
    │   ├── tester-result.txt       # 运行时结果的约定名称
    │   └── quality-result.txt      # 运行时结果的约定名称
    └── settings.json
```

读取这套结构时，先区分长期配置和一次性结果：`skills/`、`agents/`、脚本与必要的设置描述流程；`results/` 保存某一次检查后的临时状态。迁移配置时不应沿用旧工程的 PASS 文件。

这些是课程应用项目的文件。当前学习笔记目录只保存 Notebook，不需要为写笔记安装这套 `.claude` 配置。


## 4. 补充：把迁移任务说明清楚，再逐项核对

可以先复制需要复用的定义，再让 AI 逐项适配；已有目标文件时先比较内容并合并，避免整份覆盖当前工程的约定。下面是根据共享文件整理的补充迁移指令，供在课程应用项目中使用：

```text
把记账项目中的 Skills、tester、quality-engineer 和 gitcommit-agent
迁移到当前 LangChainRAG项目。

先读取目标项目的 README、backend/requirements.txt、frontend/package.json、
已有测试和 .claude 配置，再修改技能与代理定义。

需要适配：
1. 项目名称、技术栈、源码目录、测试目录和 Python 虚拟环境。
2. 前端 Vitest 与后端 pytest 各自的工作目录和真实运行命令。
3. 安全审计中的 JWT、SQLAlchemy、上传、SSE 和 RAG 检索生成流程。
4. 两份质量门结果的文件名、判定标准与生命周期。
5. 启动和重建技能中的系统命令、端口、数据目录与操作影响。

保留目标项目需要的已有设置，逐项合并 hooks。
课堂机器的绝对路径和权限列表单独核对，不直接整份覆盖。
这次只整理迁移文件并报告待验证项，不执行 Git 提交或推送。
```

迁移后可在应用根目录做一次文本核对。以下只是补充检查命令，本次没有在课程工程中执行：

```powershell
Get-ChildItem -LiteralPath .claude -Recurse -File
rg -n '黑马记账|Tauri|Dexie|IndexedDB|d:/LangChainRAG项目' .claude
```

命中旧项目词不一定就是错误，例如一个教学反例可以继续说明相同原则；需要判断它是通用例子，还是正在决定真实操作的路径、命令与技术假设。后者必须与当前工程一致。


## 5. unit-test：先把两端测试入口适配正确

迁移后的 `.claude/skills/unit-test/SKILL.md` 头部原文为：

```yaml
---
name: unit-test
description: 创建单元测试、执行测试、并生成测试报告（支持 Python 后端 + TypeScript 前端）
---
```

原技能按“确定目标 → 创建测试文件 → 执行测试 → 生成报告”组织工作。优先检查纯计算与工具函数，其次是数据转换、API 端点和 React 组件。它已经从单一前端测试流程扩展为两套测试体系。

原“可用命令”表保留了下面的写法：

| 原命令 | 原用途 | 原表标注目录 |
|---|---|---|
| `npx vitest run` | 运行所有前端测试 | `frontend/` |
| `python -m pytest backend/tests/ -v` | 运行所有后端测试 | `backend/` |

后端这一行存在目录与参数不一致的问题：已经进入 `backend/`，再传 `backend/tests/`，通常会指向不存在的 `backend/backend/tests/`。原技能步骤 3 还写有 `cd backend && python -m pytest backend/tests/ -v`，同样需要修正。

**补充：统一从应用根目录调用明确的环境与目标。**沿用 11 已核对的四个后端单元测试文件，可以写成：

```powershell
.\backend\venv\Scripts\python.exe -m pytest `
  backend/tests/test_config.py `
  backend/tests/test_security.py `
  backend/tests/test_splitter.py `
  backend/tests/test_rag_chain.py -v
```

若终端已经位于 `backend/`，则改为该目录内的路径：

```powershell
.\venv\Scripts\python.exe -m pytest `
  tests/test_config.py `
  tests/test_security.py `
  tests/test_splitter.py `
  tests/test_rag_chain.py -v
```

前端在另一个终端中从应用根目录进入 `frontend/`，使用已核对的 `package.json` 脚本：

```powershell
Set-Location -LiteralPath frontend
npm test
```

这里的 `npm test` 对应 `vitest run`。原技能使用的 `npx vitest run` 以本地依赖已安装为前提；缺少依赖时先报告环境问题。后端原依赖清单未列出 pytest，也不能把“工具不存在”归为测试通过。

本篇沿用 11 的明确收集范围，避免在后端根目录宽泛收集时导入会在顶层发起模型请求的 `test_api.py`、`test_dashscope.py`。后续增加测试文件时，应同步更新已确认的单元测试范围。


## 6. 原技能的测试模板，还需要补成真实断言

迁移后的技能给出以下 Python 模板，原文保留了 `pass`：

```python
import pytest

class TestFormatDocs:
    """格式化检索文档 — 测试套件"""

    def test_正常输入_返回格式化上下文(self):
        """输入: 有效的 Document 列表 → 输出: 格式化字符串+来源列表"""
        pass

    def test_空列表_返回空上下文(self):
        """输入: [] → 输出: ('', [])"""
        pass
```

它说明类名、场景名和测试文件的组织方式，但没有调用被测函数，也没有断言。pytest 即使执行这种空方法没有报错，也不能证明文档格式化符合预期。迁移技能时应让 Agent 读取实际实现，再用输入、输出和边界条件替换占位内容。

前端原模板如下：

```typescript
import { describe, it, expect } from 'vitest'

describe('formatFileSize — 文件大小格式化', () => {
  it('场景：1024字节 → 输出"1.0 KB"', () => {
    expect(formatFileSize(1024)).toBe('1.0 KB')
  })
  it('边界：0字节 → 输出"0 B"', () => {
    expect(formatFileSize(0)).toBe('0 B')
  })
})
```

这是原技能中的测试形状示例，片段没有提供 `formatFileSize` 的导入和实现。不能据此认定项目已存在相应工具函数或已经通过这些用例。11 中已经取得的前端实际测试位于 `frontend/src/store/`，迁移时同样要覆盖真实状态逻辑。

原技能要求每个测试只检查一个场景，使用中文描述，并覆盖正常、边界和非法输入。报告中的文件数、用例数、通过数、失败数和耗时应来自实际工具输出；模板里的 `X`、`Y`、`Z`、`W` 是待填写字段。


## 7. tester：把双端测试交给同一个专项角色

`.claude/agents/tester.md` 的头部原文为：

```yaml
---
name: tester
description: 单元测试专家——当用户需要创建单元测试、执行测试、查看测试报告时调用此 subagent。使用项目 /unit-test 技能来完成工作。
model: haiku
tools: Skill, Read, Write, Bash, Glob, Edit, Grep
---
```

这些字段属于课程的 Claude Code 角色定义。`model: haiku` 是该文件的模型选择；读取笔记不会改变当前聊天的模型。正文要求角色先通过 Skill 工具调用 `/unit-test`，文件头部没有 `skills:` 预加载字段，不能把正文约定误写成配置已自动预加载。

原角色的测试顺序为：

1. 后端纯函数与工具函数。
2. 后端 RAG 管道中检索、格式化等可隔离逻辑。
3. 前端纯函数。
4. 用 pytest + httpx 检查 FastAPI 路由。
5. 需要更多 Mock 的 React 组件。

顺序表达的是优先级，实际目标要落到当前源码。11 已读取的测试涉及 `core/security.py`、配置、分块器、引用格式化及前端 Store，不能只按角色里的 `backend/app/utils/` 示例目录寻找目标。

原角色还要求使用 pytest fixture 管理测试数据库，并举内存 SQLite 为例。这是角色对新增数据库测试的要求；11 读取的 `conftest.py` 主要处理导入路径和事件循环，没有据此证明共享工程已经具备完整的内存数据库夹具。

普通调用结束时，`tester` 向对话报告文件、用例数量与通过/失败情况；被 `gitcommit-agent` 以“质量门”模式调用时，最后再写入 `.claude/results/tester-result.txt`。如果修改了测试或业务实现，要以修改后的实际运行结果为准。


## 8. quality-engineer：适配双端源码和审查范围

`.claude/agents/quality-engineer.md` 的头部原文为：

```yaml
---
name: quality-engineer
description: 代码质量工程师——当用户需要对代码进行质量审查、安全检查、注释检查时调用。综合使用 /security-audit 和 /comments-check 技能，并额外检查错误处理、类型安全、重复代码等质量问题。
model: sonnet
tools: Skill, Read, Grep, Glob, Bash, Write
---
```

原角色按三部分形成综合报告：

| 部分 | 执行依据 | RAG 项目中的重点 |
|---|---|---|
| 安全 | `/security-audit` | 密钥、SQL、JWT、API、上传、RAG 与 Web 风险 |
| 注释 | `/comments-check` | 注释率、逻辑准确性和初学者可读性 |
| 代码质量 | Agent 正文内置规则 | 错误处理、类型安全、重复逻辑和无用代码 |

用户指定范围时按指定文件或目录审查；未指定时默认检查 `backend/app/` 和 `frontend/src/` 中最近修改的文件；“全面检查”则覆盖这两个源码目录。

后端质量检查关注数据库、嵌入、检索、LLM、ChromaDB 的失败处理，以及 FastAPI 的 `HTTPException`、Python 类型提示、Pydantic 校验和 SQLAlchemy 模型。前端关注 Promise、axios、SSE 的错误路径、输入校验、`any`、类型断言和可空数据处理。

这些规则需要结合调用链解释。例如失败已经在上层集中处理时，不能只因某个异步函数没有局部 `try-except` 就认定必须修复；应指出错误会怎样传播、谁负责回滚或通知用户。这里是对原检查规则的补充解释。

原角色默认只分析业务源码并给出修复示例；质量门模式允许它写 `.claude/results/quality-result.txt`。这项结果文件写入例外，与随意修改被审查源码是不同范围。修复发生后，还要重新审查最终内容。


## 9. security-audit：加入 RAG 与 Web 应用的真实边界

迁移后的安全技能头部为：

```yaml
---
name: security-audit
description: 安全审计：检查敏感信息泄露、SQL注入、JWT认证、API安全、Web安全等（FastAPI + React 前后端项目）
---
```

原技能正文包含六个维度：

| 维度 | 原检查重点 |
|---|---|
| 敏感信息 | 硬编码密码、百炼 API Key、JWT 密钥、带密码的连接串、管理员密码 |
| SQL 注入 | 拼接 SQL、SQLAlchemy 参数化查询、`text()` 绑定参数、动态排序白名单 |
| JWT 与认证 | 密钥、过期时间、Authorization 传递、bcrypt、角色权限、token 存储 |
| FastAPI 与 Web API | CORS、限流、错误信息、上传大小和类型、路径遍历、SSE 认证 |
| LangChain / RAG | 提示词注入、文档可信边界、ChromaDB 目录、API Key、检索数量限制 |
| 其他风险 | XSS、动态执行、安全随机数、日志、输入校验、依赖和 Git 忽略规则 |

技能中 SQL 检查的核心是判断用户输入是否进入 SQL 结构。原文将字符串拼接 SQL 列为危险示例，将 `select(User).where(User.username == username)` 列为参数化查询示例。搜索关键词只是找到线索，还需要读取调用上下文；普通文本中出现 `token` 或 `localhost`，本身不能直接确定风险等级。

`quality-engineer` 的六项摘要把“配置文件安全”单列出来，技能正文则专门增加了“LangChain / RAG 安全”。调用时应读取技能正文，结合配置检查一起做，避免只照角色摘要遗漏 RAG 文档与提示词边界。

**补充：默认源码范围之外也要核对提交文件。**11 读取的 `backend/test_api.py`、`backend/test_dashscope.py` 位于 `backend/app/` 外，原共享脚本包含硬编码凭据和相关打印逻辑。本篇不复制原密钥；这个现象说明只扫描 `backend/app/ + frontend/src/`，可能遗漏将被提交的辅助脚本。

RAG 的专门问题是：检索文档提供回答依据，其中的文字也可能带有指令。检查时应确认系统如何把文档作为资料使用，不能仅凭提示词出现“引用知识库”就认定已经解决提示词注入。API Key 的配置与日志也需要单独核对。

原技能默认只出审计报告，报告包含风险等级、位置、原因和修复建议；“项目用于本地毕业设计”是风险评估背景，不能代替实际数据流与权限检查。


## 10. comments-check：同时检查 Python 与 TypeScript

迁移后的注释技能头部为：

```yaml
---
name: comments-check
description: 检查代码注释质量：覆盖率、准确性、小白可读性。支持 Python 后端 + TypeScript 前端
---
```

原技能检查三个维度：

| 维度 | 原要求 |
|---|---|
| 覆盖率 | 每 10 行代码至少 3 行注释，目标至少 30%；空白行不计数 |
| 准确性 | 注释中的行为、参数和返回类型与当前实现一致，标出过期注释 |
| 可读性 | 用通俗语言解释用途；Python docstring 开头应说明函数做什么 |

它已补充 Python 的 `#`、docstring 与 TypeScript 的行注释、块注释、JSDoc，并要求 Python 类和模块顶部有说明。原技能里的向量化 docstring 示例用“把问题转换成数字向量”解释用途；其中维度等具体数值仍应与实际模型配置一致。

需要区分技能目标和质量门阈值：

| 文件中的位置 | 原数值 | 含义 |
|---|---:|---|
| `comments-check/SKILL.md` | 30% | 注释检查的目标 |
| `quality-engineer.md` 普通检查规则 | 30% | 审查说明中的目标 |
| `quality-engineer.md` Gate Mode | 20% | 当前快照决定放行的最低阈值 |

因此 20%—29% 可能满足原质量门的注释条件，却没有达到技能的 30% 目标。迁移时应明确是否保留这种“目标与底线”的区别；如果要求统一标准，就需要同步修改技能、角色和报告中的数值。不能在笔记中默默把原文件改成同一个比例。

注释率的分母、混合代码注释行和多行 docstring 的统计方式也要一致。比例达到目标后，仍需检查准确性，避免用重复代码含义的注释凑数。原技能默认只给报告和具体改进示例。


## 11. run-app 与 rebuild-app：运行流程也要随项目迁移

共享工程还包含两项运行技能：

| 技能 | 原 description | 当前应用目标 |
|---|---|---|
| `run-app` | 启动电商RAG知识库问答系统（开发模式），用于预览和测试 | 启动 FastAPI 与 Vite，检查页面与 API 文档 |
| `rebuild-app` | 清理并重建项目依赖与数据库（重置环境） | 重建数据库、向量库、Python 环境与前端依赖 |

`run-app` 推荐 `start.bat`，并提供手动启动方式：后端使用 `python -m uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload`；前端使用 `npm run dev`。原技能标注后端 8000、前端 5173、API 文档 `http://localhost:8000/docs`，API Key 位于 `backend/.env` 的 `DASHSCOPE_API_KEY`。

原文把 `venv\Scripts\activate` 放在 `bash` 围栏内，这实际上带有 Windows CMD 的写法。**补充：在 PowerShell 中直接调用虚拟环境解释器，减少激活脚本和 Shell 的歧义。**在应用根目录打开两个终端：

```powershell
# 终端一：后端
Set-Location -LiteralPath backend
.\venv\Scripts\python.exe -m uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

```powershell
# 终端二：前端
Set-Location -LiteralPath frontend
npm run dev
```

启动后应核对实际控制台显示的地址，Vite 在端口占用时可能选择其他端口。原技能还保留了课程默认管理员 `admin / 123456`；实际登录行为取决于初始化与当前配置。这里记录技能内容，本次没有执行 `start.bat` 或登录应用。

`rebuild-app` 原文首先执行：

```bash
rm -rf backend/data/app.db
rm -rf backend/data/chromadb/
```

随后删除并重建 `backend/venv/`，安装 `requirements.txt`，再删除 `frontend/node_modules/` 并运行 `npm install`。原技能已说明这会删除数据库记录与向量索引，需要重新注册用户、上传文档并索引；这是开发环境的重置操作。

迁移这项技能时必须保留操作影响说明。数据库里的用户、会话、消息和文档记录会受到重置影响；上面两行没有直接删除 `uploads/`，不能把它写成已清理所有上传文件。原步骤还先 `cd backend`，后直接 `cd frontend`，连续在同一个终端执行会出现兄弟目录定位问题，应从明确的应用根目录分别进入。

`run-app` 中“ChromaDB 错误就删除向量目录”的排查建议也涉及索引丢失；应先核对错误、版本和数据可重建条件。把技能文件迁移过来，只代表流程已保存，不代表每次故障都适合重置。


## 12. gitcommit-agent：组织两项检查，再进入 Git 操作

提交代理的原始头部为：

```yaml
---
name: gitcommit-agent
description: Git 提交质量守门员——在提交代码前自动运行单元测试和代码质量检查，两者全部通过才允许提交并推送。当用户说"帮我提交代码"、"提交"、"commit"等时自动调用。
model: sonnet
tools: Agent, Read, Bash, Write, Glob
---
```

它使用 `Agent` 工具组织两个专项角色。原流程先清理旧结果，再依次调用测试专家和质量工程师，随后读取结果决定是否提交：

```bash
rm -f .claude/results/tester-result.txt .claude/results/quality-result.txt
mkdir -p .claude/results
```

原文件交给 `tester` 的 prompt 为：

```text
质量门模式 — 请运行项目的所有单元测试，并将结果写入 .claude/results/tester-result.txt。前端用 npx vitest run 测，后端用 python -m pytest backend/tests/ -v 测（如果有 pytest）。如果所有测试通过写 PASS，否则写 FAIL 并列出失败明细。
```

交给 `quality-engineer` 的 prompt 为：

```text
质量门模式 — 请对项目进行全面的代码质量检查（安全审计 + 注释检查 + 代码质量审查），并将结果写入 .claude/results/quality-result.txt。三个维度全部通过写 PASS，否则写 FAIL 并列出问题。
```

原代理随后使用 `cat` 读取两个文件。两者第一行都是 `PASS` 才进入提交流程；任一 `FAIL` 或文件缺失，都报告失败详情，清理临时结果，并要求修复后重新检查。

**补充：迁移时要收紧“如果有 pytest”的含义。**当前项目有前后端两套检查要求，缺少工具、导入错误或没有收集到预期测试时应报告阻塞，不能跳过后端后用前端通过结果生成整体 PASS。两端的工作目录、运行命令、退出码和实际收集范围都要记录。

原通过分支包含以下 Git 操作，保留原命令与多行提交信息的形状：

```bash
git add .
git log --oneline -5
git commit -m "提交信息

Co-Authored-By: Claude <noreply@anthropic.com>"
git push
```

其中 `git log --oneline -5` 用于参考已有提交格式；原代理要求查看改动摘要并生成中文提交信息。上述命令是课程流程，当前笔记整理没有执行暂存、提交或推送。实际提交与推送应处于用户明确要求处理的仓库和操作范围内。


## 13. Gate Mode：两份结果各自证明什么

普通调用只把报告输出到对话；prompt 包含“质量门”并由提交代理组织时，两个角色在检查结束的最后一步写结果文件。

| 结果文件 | 原 PASS 条件 |
|---|---|
| `.claude/results/tester-result.txt` | 要求执行的测试全部通过，退出码为 0 |
| `.claude/results/quality-result.txt` | 安全 0 高危、0 中危；注释率至少 20%；没有必须修复的异常或严重类型问题 |

通过时原文件第一行为：

```text
PASS
```

失败时第一行写 `FAIL`，后面附失败明细。例如下面是字段模板，数字仍需来自实际检查：

```text
FAIL
测试文件：X 个
测试用例：Y 个
通过：Z 个
失败：W 个
失败明细：
  [文件路径、用例名、期望行为与实际结果]
```

原提交代理规定开始清理旧文件、结束销毁结果，以降低重复消费旧 PASS 的可能。缺少文件、只生成一份文件、首行不是约定结果，都不应进入提交分支。

还要检查结果对应的内容。测试通常读取工作区，而 Git 提交通常读取暂存区；检查后又改了代码，或者工作区修复没有进入暂存区，原 PASS 就不能说明将要提交的版本符合条件。迁移时应保证检查范围与提交范围一致，内容变化后重新检查。

原文件没有把 PASS 与具体代码版本绑定，也没有把完整运行证据写进 PASS 行。**补充：报告中保留两端命令、工作目录、退出码与摘要，消费结果前核对最终 diff。**这些文本标记是流程状态，不能代替真实测试和审查过程。


## 14. Bash 与 PowerShell 脚本：相同意图，比较细节有差异

原 `.claude/scripts/check-quality-gate.sh` 使用相对路径，核心逻辑如下；省略了重复的提示横线与详细报告打印，保留判定和退出码：

```bash
RESULTS_DIR=".claude/results"
TESTER_FILE="$RESULTS_DIR/tester-result.txt"
QUALITY_FILE="$RESULTS_DIR/quality-result.txt"

if [ ! -f "$TESTER_FILE" ] || [ ! -f "$QUALITY_FILE" ]; then
  exit 2
fi

TESTER_RESULT=$(head -1 "$TESTER_FILE")
QUALITY_RESULT=$(head -1 "$QUALITY_FILE")

if [ "$TESTER_RESULT" = "PASS" ] && [ "$QUALITY_RESULT" = "PASS" ]; then
  exit 0
else
  exit 2
fi
```

原 PowerShell 脚本的对应路径、存在性与首行读取为：

```powershell
$ResultsDir = ".claude/results"
$TesterFile = Join-Path $ResultsDir "tester-result.txt"
$QualityFile = Join-Path $ResultsDir "quality-result.txt"

$testerExists = Test-Path $TesterFile -PathType Leaf
$qualityExists = Test-Path $QualityFile -PathType Leaf

if (-not $testerExists -or -not $qualityExists) {
    exit 2
}

$testerResult = (Get-Content $TesterFile -First 1).Trim()
$qualityResult = (Get-Content $QualityFile -First 1).Trim()

if ($testerResult -eq "PASS" -and $qualityResult -eq "PASS") {
    exit 0
}
else {
    exit 2
}
```

两段均为原文件核心摘录，并非本次执行结果。完整脚本还打印缺失、失败内容及提示。

迁移到 Windows 时需要注意以下差异：

| 细节 | Bash 原脚本 | PowerShell 原脚本 |
|---|---|---|
| 首行空白 | 与 `PASS` 精确比较 | 先 `.Trim()`，会忽略首尾空白 |
| 大小写 | 区分 `PASS` 与 `pass` | `-eq` 默认不区分大小写 |
| CRLF | `head` 结果可能残留 `\r`，影响比较 | `Get-Content` 按行读取，通常不保留换行字符 |
| 空文件 | 首行为空，进入失败分支 | 首行可能为 `$null`，调用 `.Trim()` 会报错 |
| 工作目录 | 从当前目录找 `.claude/results` | 同样依赖当前目录 |

因此“首行恰好为 PASS”的约定，在两份原脚本中并不完全等价。补充完善时应明确编码、大小写和空白规则，处理读取失败及空文件；严格区分大小写时 PowerShell 可使用 `-ceq`。

脚本存在也不代表它已接入 Hook。当前共享 `settings.json` 使用的是下一节的 `type: agent`，没有引用这两个脚本作为 command Hook。选择脚本路线时，还需要配置事件、命令、工作目录和输入解析；相关机制见 09，本篇不再另建一套实现。


## 15. settings.json：合并路径、权限与 Hook 协议

共享 `settings.json` 的 `additionalDirectories` 写有课堂机器的 `d:/LangChainRAG项目/.claude`、`backend`、`frontend` 和运行数据目录；权限列表包含 npm、Python、Git 及文件操作相关规则。迁移时应逐项核对目标路径、工具和任务需要，避免用课堂机器的设置覆盖已有工程配置。

下面只保留原文件 `hooks` 部分，按 JSON 重新排版，字段和值保持原意：

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Bash",
        "hooks": [
          {
            "type": "agent",
            "prompt": "You are a git commit quality gate. Use Read tool to check two files: .claude/results/tester-result.txt and .claude/results/quality-result.txt. If BOTH files exist AND BOTH first lines are exactly 'PASS': return {\"hookSpecificOutput\": {\"permissionDecision\": \"allow\"}, \"systemMessage\": \"Quality gate PASS - commit allowed\"}. Otherwise: return {\"hookSpecificOutput\": {\"permissionDecision\": \"deny\"}, \"systemMessage\": \"Quality gate BLOCKED - run gitcommit-agent first to generate pass markers at .claude/results/\"}. You MUST actually read the files with Read tool before deciding.",
            "if": "Bash(*git commit*)",
            "timeout": 30
          }
        ]
      }
    ]
  }
}
```

| 字段 | 当前配置表达的意图 |
|---|---|
| `PreToolUse` | 在工具执行前检查 |
| `matcher: Bash` | 选择 Bash 工具 |
| `if: Bash(*git commit*)` | 用权限规则形式进一步过滤提交命令 |
| `type: agent` | 启动有工具访问能力的验证代理 |
| `prompt` | 要求实际读取两份结果，检查首行 |
| `timeout: 30` | 本例显式设置的超时秒数 |

这项 Hook 验证代理与 `gitcommit-agent.md` 是两个入口：提交代理组织检查和 Git 操作，Hook 在命中的工具事件上核对是否已有结果。

**接口核对：**当前 [Hooks reference](https://code.claude.com/docs/en/hooks#agent-based-hooks) 要求 agent Hook 返回 `ok`，通过与阻塞的结构分别为：

```json
{"ok": true}
```

```json
{"ok": false, "reason": "缺少本次测试结果，请重新完成质量检查"}
```

原 prompt 要求返回 `hookSpecificOutput.permissionDecision`，混用了 command Hook 的事件输出协议。迁移时至少应按 agent 类型修正这项返回约定；它在当前文档中属于实验能力，原配置不能直接写成已验证生效。

**补充的 prompt 片段**可表达为：

```text
检查此次提交所在应用项目的 .claude/results/tester-result.txt 和
.claude/results/quality-result.txt。事件输入：$ARGUMENTS。
必须实际读取两份文件；都可读且首行符合 PASS 约定时返回 {"ok": true}；
否则返回 {"ok": false, "reason": "说明具体缺失或失败项"}。
只验证结果，不改写结果文件，不提交代码。
```

这里的 `$ARGUMENTS` 是 agent Hook 的事件输入占位符。只匹配 Bash 的配置不会覆盖原生 PowerShell 等其他工具中的提交，也不是仓库级 Git Hook；覆盖范围与实际拦截需要按 09 的方法验证。返回格式修正后，仍需解决相对路径与旧 PASS 的有效期问题。


## 16. 提交前核对 .gitignore 与暂存内容

RAG 工程根目录 `.gitignore` 的相关原规则如下，省略了 IDE 与操作系统文件部分：

```gitignore
# Python
__pycache__/
*.py[cod]
*.so
venv/
*.egg-info/
dist/
build/

# Frontend
node_modules/
frontend/dist/

# Runtime data
backend/data/
!backend/data/.gitkeep

# Environment
.env
!.env.example
```

它的意图是排除依赖、构建产物、运行数据与实际环境配置。忽略规则不会自动移除已经跟踪的文件，也不会发现源码中硬编码的密钥；`.env.example` 本身仍需检查是否只包含示例值。

原规则有两处与本篇直接相关的边界：

1. `backend/data/` 排除了整个父目录，下面的 `!backend/data/.gitkeep` 通常不能重新纳入该子文件。若需要保留占位文件，应调整目录忽略方式。
2. 原文件没有忽略 `.claude/results/`。原代理又使用 `git add .`，所以提交前生成的临时通行证可能进入暂存区。

**补充规则示例：**需要保留占位文件时，可把原运行数据两行改为下面的写法，并追加临时结果目录；这是建议，未修改课程工程：

```gitignore
backend/data/*
!backend/data/.gitkeep
.claude/results/
```

“提交后删除通行证”不能撤回已经进入提交历史的文件，所以暂存检查应发生在提交之前。对已有跟踪文件，还需根据实际状态处理，不能仅靠新增忽略规则判断已经排除。

在课程应用根目录，可以先用以下补充命令检查分支、远端和改动：

```powershell
git status --short
git branch --show-current
git remote -v
git diff --stat
git diff -- .gitignore .claude
```

确认本次范围后，按实际文件显式暂存。例如本次确实包含迁移配置时，可选这些目录和文件，其他业务或测试改动另按审查结果加入：

```powershell
git add -- .claude/skills .claude/agents .claude/scripts .claude/settings.json .gitignore
git diff --cached --stat
git diff --cached --check
git diff --cached
```

提交前应确认暂存内容与检查内容一致，没有夹带环境文件、运行数据和临时结果。当前可读取资料没有提供本节真实提交哈希与远端配置，因此这里不填写仓库地址、分支名或假定首次推送参数；原 `git push` 是否具备 upstream 也需要在实际仓库中核对。


## 17. 补充：验证迁移是否真正接上流程

文件齐全后，还要分别验证发现、执行和拦截。可在课程应用根目录启动 Claude Code，再在会话内查看技能与角色：

```text
/skills
/agents
/unit-test 先报告前后端测试入口、工作目录和依赖状态
```

这些是 Claude Code 会话入口，不是 PowerShell 命令。技能路径、frontmatter 和代理名的基本机制沿用 07—08；发现结果要以当前会话实际显示为准。

下表是补充验证项，本次仅用于学习，未在应用中运行：

| 场景 | 应取得的证据 |
|---|---|
| 发现技能与角色 | 当前项目能识别五项技能及三项角色，来源路径正确 |
| 调用测试专家 | 实际使用 `unit-test`，两端命令与工作目录正确，报告真实收集与退出情况 |
| pytest 缺失或导入失败 | 明确失败或阻塞原因，测试结果不能写成整体 PASS |
| 普通审查 | 产生范围、风险与位置明确的报告，保持业务源码只读 |
| 质量门审查 | 完成全部检查后，最后写入两份对应当前内容的结果 |
| 缺一份结果、空结果或 FAIL | 验证 Hook 的实际阻塞行为及说明 |
| 两份有效 PASS | 在明确的提交任务中核对命中与放行；不据此跳过最终暂存检查 |
| 检查后又修改代码 | 重新检查，避免消费旧 PASS |
| 从子目录或另一种工具提交 | 核对相对路径与工具匹配是否覆盖该入口 |
| 清理结束 | 临时结果没有被纳入提交，下一次不能沿用旧结果 |

如果测试专家修改了测试或代码，质量审查应读取这些修改完成后的版本；审查建议落实后，再对相关测试和最终代码复核。共享提交代理按步骤组织这两项检查，不能只因两个角色都输出了“完成”就跳过结果核对。

迁移后的最小闭环是：确定本次内容 → 使用正确环境测试 → 审查最终源码与配置 → 核对两份结果及暂存 diff → 在提交授权范围内提交与推送 → 清理临时状态。并发容量与性能优化需要下一节另外取得压力测试证据。


## 18. 来源与本次整理状态

课程与文件来源：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：核对第 23 节准确标题及前后章节边界。
- [Part3 LangChain RAG项目实战](https://www.yuque.com/xxcls/vibecoding/mtwqfc61tifo9cgi)：重新核对讲义正文仍以初始化提示词为主。
- 网盘 `项目/LangChainRAG项目/.claude/skills/`：读取 `unit-test`、`security-audit`、`comments-check`、`run-app`、`rebuild-app` 五份 `SKILL.md` 的源码模式全文。
- 同项目 `.claude/agents/`：读取 `tester.md`、`quality-engineer.md`、`gitcommit-agent.md` 全文。
- 同项目 `.claude/settings.json` 与 `.claude/scripts/check-quality-gate.sh`：读取原文，核对 Hook 与脚本没有直接接线。
- 同项目 `.claude/scripts/check-quality-gate.ps1` 和根目录 `.gitignore`：通过单文件下载读取原文；脚本只做语法解析，没有执行。
- 07—11 中已经核对的记账项目机制、RAG 测试、依赖清单、前端脚本与源码，用于确认迁移时的项目差异。

接口核对：

- [Claude Code Hooks reference](https://code.claude.com/docs/en/hooks)：`if` 条件、agent Hook 的事件输入与 `ok` 返回协议，核对日期 `2026-10-10`。

原定义、核心命令和判断规则与补充建议已分别标明。本次只新增 12 Notebook，没有改动 01—11、课程应用源码、技能安装目录或当前工具配置；没有运行课程测试、启动或重置应用、发起模型请求、执行 Git 暂存、提交或推送。文中的 PASS、FAIL 与报告数字是约定或模板，不是本次应用运行结果。
